# GenLab — Launcher
Plataforma modular para ejecutar modelos open source de IA en Colab.

**Antes de ejecutar:**
1. Runtime → Cambiar tipo de entorno de ejecución → **T4 GPU**.
2. Consigue tu **HF_TOKEN** (ver celda 5).
3. Concede permisos de Drive cuando se solicite.

**IMPORTANTE:** Ejecuta las celdas EN ORDEN, de arriba hacia abajo. Nunca saltes celdas.

In [ ]:
# 1. Instalar dependencias
!pip install -q torch "diffusers>=0.33.0" transformers huggingface-hub imageio imageio-ffmpeg accelerate psutil pyyaml hf_transfer

In [ ]:
# 2. Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Clonar repo, cargar genlab y bootstrap
import os, sys

REPO_URL = 'https://github.com/ke1npro/GenLab.git'
GENLAB_SRC = '/content/genlab'

# Clonar o actualizar
if os.path.isdir(GENLAB_SRC):
    !git -C $GENLAB_SRC pull
else:
    !git clone $REPO_URL $GENLAB_SRC

# Verificar que el clone funcionó
assert os.path.isdir(f'{GENLAB_SRC}/src/genlab'), \
    f'No se encontró src/genlab/ en {GENLAB_SRC} — el clone falló'

# Agregar src/ al path de Python y cambiar al repo
sys.path.insert(0, f'{GENLAB_SRC}/src')
os.chdir(GENLAB_SRC)

# Importar y bootstrap
from genlab import GenLab
print('genlab importado OK')

info = GenLab.bootstrap()

# Checklist visual
hw = info['hardware']
print()
print('--- Checklist ---')
print(f'[{"OK" if hw["has_gpu"] else "XX"}] GPU: {hw["gpu"]}')
print(f'[{"OK" if info["hf_token_ok"] else "XX"}] HF_TOKEN: {"configurado" if info["hf_token_ok"] else "pendiente"}')
print(f'[{"OK" if info.get("hf_transfer") else "--"}] hf_transfer: {"Disponible" if info.get("hf_transfer") else "No instalado"}')
print(f'[{"OK"}]          genlab importado')

In [ ]:
# 4. Token de Hugging Face
# ¿Dónde conseguirlo?
# 1. https://huggingface.co/join — crear cuenta
# 2. https://huggingface.co/settings/tokens — New token → Role: Read
# 3. Copia el token (empieza con 'hf_')
# 4. Si usas FLUX: https://huggingface.co/black-forest-labs/FLUX.1-dev — requiere aceptar términos

try:
    from genlab import GenLab
except ModuleNotFoundError:
    raise RuntimeError('genlab no está cargado. Ejecuta primero la CELDA 3 (la anterior). Si ya la ejecutaste, haz Runtime → Restart and run all para empezar fresco.')

from google.colab import userdata
import os
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('[OK] HF_TOKEN cargado desde Secrets de Colab.')
except Exception:
    token = input('Pega tu HF_TOKEN (o Enter para omitir): ').strip()
    if token:
        os.environ['HF_TOKEN'] = token
        print('[OK] HF_TOKEN configurado manualmente.')
    else:
        print('[XX] HF_TOKEN no configurado.')

In [ ]:
# 5. Elegir modelo y tarea

MODELOS = [
    {"tag": "wan",       "label": "Wan2.1 T2V 1.3B  (video)",             "default_model_id": "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"},
    {"tag": "cogvideo",  "label": "CogVideoX-2b  (video, ~9GB)",         "default_model_id": "zai-org/CogVideoX-2b"},
    {"tag": "flux",      "label": "FLUX.1-schnell  (imagen, 4 pasos)",   "default_model_id": "black-forest-labs/FLUX.1-schnell"},
    {"tag": "sdxl",      "label": "SDXL 1.0  (imagen, 1024x1024)",       "default_model_id": "stabilityai/stable-diffusion-xl-base-1.0"},
    {"tag": "ssd1b",     "label": "SSD-1B  (imagen, ~3GB, destilado SDXL)", "default_model_id": "segmind/SSD-1B"},
]

print("Modelos disponibles:")
print("  0. Custom  (cualquier modelo de HuggingFace)")
for i, m in enumerate(MODELOS, 1):
    print(f"  {i}. {m['label']}")

seleccion = input("\nElige el n\u00famero del modelo [1]: ").strip()

if seleccion == "0" or not seleccion.isdigit():
    # Custom: pedir model_id y auto-detectar provider
    custom_id = input("Model ID de HuggingFace (ej. stabilityai/stable-diffusion-xl-base-1.0): ").strip()
    while not custom_id:
        custom_id = input("Debes ingresar un Model ID: ").strip()
    print(f"[..] Detectando pipeline para {custom_id}...")
    from genlab.models import detect_provider
    prov = detect_provider(custom_id)
    if prov:
        modelo_final = prov
        print(f'[OK] Provider detectado: {prov}')
    else:
        # fallback: si tiene "flux" en el nombre, asumir flux; si no, asumir sdxl
        modelo_final = "flux" if "flux" in custom_id.lower() else "sdxl"
        print(f'[!!] No se pudo detectar autom\u00e1ticamente. Asumiendo: {modelo_final}')
    config_extra = {'model': {'model_id': custom_id}}
    tarea = "text_to_image" if modelo_final in ("flux", "sdxl", "ssd1b") else "text_to_video"
    print(f'[OK] Modelo custom: {custom_id}')
    print(f'[OK] Tarea: {tarea}')
else:
    idx = int(seleccion) - 1 if seleccion.isdigit() and 1 <= int(seleccion) <= len(MODELOS) else 0
    modelo_info = MODELOS[idx]
    modelo_final = modelo_info["tag"]
    config_extra = {}
    print(f'\n[OK] Modelo: {modelo_info["label"]} ({modelo_info["default_model_id"]})')
    tarea = "text_to_image" if modelo_final in ("flux", "sdxl", "ssd1b") else "text_to_video"
    print(f'[OK] Tarea: {tarea}')

In [ ]:
# 6. Hilos de descarga
# más hilos = descarga más rápida, pero más uso de CPU/RAM

GENLAB_MAX_WORKERS = 6  # cambia este valor si quieres (4, 8, 16...)
import os
os.environ['GENLAB_MAX_WORKERS'] = str(GENLAB_MAX_WORKERS)
print(f'[OK] Hilos de descarga: {GENLAB_MAX_WORKERS}')

In [ ]:
# 7. Generar
# Descarga el modelo (primera vez), carga en GPU y genera.

try:
    from genlab import GenLab
except ModuleNotFoundError:
    raise RuntimeError('genlab no está cargado. Ejecuta primero la CELDA 3.')

prompt = input("Prompt (Enter para usar el default): ").strip() or "Un astronauta montando un caballo en la luna, estilo cinematogr\u00e1fico"

print(f'[..] Iniciando descarga de {modelo_final}...')

result = GenLab().run(
    task=tarea,
    model=modelo_final,
    prompt=prompt,
    config=config_extra,
)

if result.get("image_path"):
    print(f'[OK] Imagen generada: {result["image_path"]}')
elif result.get("video_path"):
    print(f'[OK] Video generado: {result["video_path"]}')
else:
    print(f'[XX] No se gener\u00f3 salida. Revisa el manifiesto: {result.get("manifest")}')

In [ ]:
# 8. Mostrar resultado
if result.get("image_path"):
    from IPython.display import Image
    display(Image(result['image_path'], width=512))
elif result.get("video_path"):
    from IPython.display import Video
    display(Video(result['video_path'], width=480))
else:
    print("[XX] No se encontr\u00f3 resultado para mostrar.")